# TrustLab Baseline Analysis

This notebook computes baseline behavioral metrics from TrustLab event logs.

Outputs:
- reliance rate by condition
- override rate by condition
- mean latency by condition
- two-proportion z-test for acceptance-rate difference (A vs B)


In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import pandas as pd

DATA_CANDIDATES = [
    Path("../sample_output.csv"),
    Path("sample_output.csv"),
    Path("../backend/data/events.csv"),
]

csv_path = next((p for p in DATA_CANDIDATES if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(f"No CSV found in: {DATA_CANDIDATES}")

print(f"Using dataset: {csv_path.resolve()}")
df = pd.read_csv(csv_path)
df.head()


In [ ]:
required = {"participant_id", "condition", "decision", "timestamp", "latency_ms"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df = df.copy()
df["condition"] = df["condition"].astype(str).str.strip()
df["decision"] = df["decision"].astype(str).str.strip().str.lower()
df["latency_ms"] = pd.to_numeric(df["latency_ms"], errors="coerce")
df = df.dropna(subset=["latency_ms"])

df["accept"] = (df["decision"] == "accept").astype(int)

attention_df = pd.DataFrame()
if "is_attention_check" in df.columns:
    attn_mask = df["is_attention_check"].fillna(False).astype(bool)
    attention_df = df[attn_mask].copy()
    df = df[~attn_mask].copy()

overall = {
    "n_events": int(len(df)),
    "n_participants": int(df["participant_id"].nunique()),
    "mean_latency_ms": float(df["latency_ms"].mean()) if len(df) else float("nan"),
    "reliance_rate": float(df["accept"].mean()) if len(df) else float("nan"),
    "override_rate": float(1.0 - df["accept"].mean()) if len(df) else float("nan"),
}
if not attention_df.empty and "attention_check_passed" in attention_df.columns:
    overall["attention_check_pass_rate"] = float(attention_df["attention_check_passed"].fillna(False).astype(int).mean())

by_condition = (
    df.groupby("condition", as_index=False)
      .agg(
          n_events=("decision", "size"),
          n_participants=("participant_id", "nunique"),
          reliance_rate=("accept", "mean"),
          mean_latency_ms=("latency_ms", "mean"),
      )
)
by_condition["override_rate"] = 1.0 - by_condition["reliance_rate"]

overall, by_condition.sort_values("condition")


In [ ]:
def two_proportion_z_test(success_a: int, total_a: int, success_b: int, total_b: int):
    if min(total_a, total_b) == 0:
        return float("nan"), float("nan")

    p_pool = (success_a + success_b) / (total_a + total_b)
    se = math.sqrt(p_pool * (1 - p_pool) * (1 / total_a + 1 / total_b))
    if se == 0:
        return float("nan"), float("nan")

    p_a = success_a / total_a
    p_b = success_b / total_b
    z = (p_a - p_b) / se
    p_two_tailed = math.erfc(abs(z) / math.sqrt(2))
    return z, p_two_tailed

if {"A", "B"}.issubset(set(df["condition"])):
    a = df[df["condition"] == "A"]
    b = df[df["condition"] == "B"]
    z, p = two_proportion_z_test(int(a["accept"].sum()), len(a), int(b["accept"].sum()), len(b))
    print({
        "A_n": len(a),
        "B_n": len(b),
        "A_reliance_rate": float(a["accept"].mean()),
        "B_reliance_rate": float(b["accept"].mean()),
        "z_score": z,
        "p_value_two_tailed": p,
    })
else:
    print("Both conditions A and B are required for between-condition test.")


In [ ]:
summary = by_condition.sort_values("condition").copy()
summary["reliance_rate"] = summary["reliance_rate"].round(4)
summary["override_rate"] = summary["override_rate"].round(4)
summary["mean_latency_ms"] = summary["mean_latency_ms"].round(2)
summary


## Interpretation Checklist

- Verify that both conditions have non-trivial sample sizes.
- Check reliance-rate directionality against the cue hypothesis.
- Inspect latency differences for speed/hesitation effects.
- If sample sizes are small, treat p-values as provisional.
